# Understanding Self-Attention: A Simplified Walkthrough

This notebook demonstrates a simplified version of the self-attention mechanism, a core component in many modern NLP models like Transformers. We will walk through the process of calculating context vectors for words in a sentence.

## 1. Setup and Token Embeddings

First, we import the necessary library (`torch`) and define our input. We'll represent each word (token) in our sentence as a numerical vector. For simplicity, we'll consider the sentence "Hello my name is Ahtesham" and represent each of its 5 words with a 3-dimensional vector.

In [14]:
import torch

# Define the example sentence (implicitly, as we're just creating embeddings)
sentence = "Hello my name is Ahtesham"
num_tokens = 5
embedding_dim = 3

# Create a tensor of shape (num_tokens x embedding_dim) to represent
# the initial, non-contextualized embedding vectors for each word.
# We use random values for demonstration purposes.
token_embeddings = torch.rand(num_tokens, embedding_dim)

print("Initial Embedding Vectors for each word:")
print(token_embeddings)
print("\nShape of the embedding vectors tensor:")
print(token_embeddings.shape)

Initial Embedding Vectors for each word:
tensor([[0.6879, 0.5702, 0.6035],
        [0.4579, 0.1599, 0.2217],
        [0.8721, 0.6651, 0.3130],
        [0.5927, 0.0789, 0.1489],
        [0.0102, 0.1318, 0.5137]])

Shape of the embedding vectors tensor:
torch.Size([5, 3])


## 2. Calculating Attention Scores

Attention scores determine how much each word in the input sentence relates to every other word. A higher score indicates a stronger relationship. Here, we calculate these scores using a dot product between each word's embedding (acting as a 'query') and every other word's embedding (acting as 'keys').

We will create a 5x5 matrix to store these scores, where `attention_score[i,j]` represents the score of word `i` attending to word `j`.

### Time Complexity for Dot Products

-   **Loop-based approach**: If we have `N` tokens and each embedding has a dimension `D`, calculating the dot product for each pair of `N` tokens involves `N * N` dot products. Each dot product takes `O(D)` time. So, the total time complexity for the nested loops is **O(N^2 * D)**.
-   **Matrix Multiplication approach**: A more efficient way to compute all dot products simultaneously is using matrix multiplication. Specifically, `token_embeddings @ token_embeddings.T`. The time complexity for multiplying two matrices of sizes `N x D` and `D x N` (which is `token_embeddings` and `token_embeddings.T`) is typically **O(N^2 * D)** or **O(N * D^2 + N^2 * D)** depending on the implementation and relative sizes, but in general, highly optimized linear algebra libraries perform this much faster than explicit loops due to underlying C/CUDA implementations and parallelization. For square matrices, it's often considered **O(N^3)** but for our specific case where `D` can be smaller than `N`, `O(N^2 * D)` is a more accurate tight bound, and it's significantly faster in practice.

In [15]:
# Initialize a 5x5 empty matrix to hold the attention scores using loops.
# Each row 'i' corresponds to a 'query' word, and each column 'j' corresponds to a 'key' word.
attention_score_loops = torch.empty(num_tokens, num_tokens)

# Loop through each word's embedding (as a 'query')
for i, x_i in enumerate(token_embeddings):
    # For each query word, calculate its similarity with every other word (as 'keys')
    for j, x_j in enumerate(token_embeddings):
        # The dot product measures the similarity between two vectors.
        attention_score_loops[i,j] = torch.dot(x_i, x_j)

print("Calculated Attention Scores (dot products via loops):")
print(attention_score_loops)

# --- More efficient way using matrix multiplication ---
# The dot product of each row vector with every other row vector can be computed
# by multiplying the token_embeddings matrix by its transpose.
# (N x D) @ (D x N) = (N x N)
attention_score_matmul = token_embeddings @ token_embeddings.T

print("\nCalculated Attention Scores (dot products via matrix multiplication):")
print(attention_score_matmul)

# Verify that both methods produce approximately the same results
print("\nAre the results from loops and matrix multiplication approximately equal?")
print(torch.allclose(attention_score_loops, attention_score_matmul))

# We will use the matrix multiplication result for subsequent steps due to its efficiency.
attention_score = attention_score_matmul

Calculated Attention Scores (dot products via loops):
tensor([[1.1626, 0.5400, 1.1680, 0.5426, 0.3922],
        [0.5400, 0.2844, 0.5751, 0.3171, 0.1396],
        [1.1680, 0.5751, 1.3009, 0.6160, 0.2573],
        [0.5426, 0.3171, 0.6160, 0.3798, 0.0929],
        [0.3922, 0.1396, 0.2573, 0.0929, 0.2814]])

Calculated Attention Scores (dot products via matrix multiplication):
tensor([[1.1626, 0.5400, 1.1680, 0.5426, 0.3922],
        [0.5400, 0.2844, 0.5751, 0.3171, 0.1396],
        [1.1680, 0.5751, 1.3009, 0.6160, 0.2573],
        [0.5426, 0.3171, 0.6160, 0.3798, 0.0929],
        [0.3922, 0.1396, 0.2573, 0.0929, 0.2814]])

Are the results from loops and matrix multiplication approximately equal?
True


## 3. Normalizing Attention Scores to Attention Weights

Raw attention scores can be any real number. To use them as weights for combining information, we need to normalize them so they sum up to 1 for each query word. This is typically done using the `softmax` function. The output values, called 'attention weights', can be interpreted as probabilities, indicating the proportion of attention a word should give to other words.

We apply `softmax` along `dim=-1` (the last dimension), meaning that for each row (each query word), the values will sum to 1.

In [16]:
# Apply softmax to normalize the attention scores into attention weights.
# `dim=-1` ensures normalization happens across each row, so each query's weights sum to 1.
attention_weights = torch.softmax(attention_score, dim=-1)

print("Normalized Attention Weights:")
print(attention_weights)

# Verify that the sum of weights for the first query word is 1
print(f"\nSum of attention weights for the first word: {attention_weights[0].sum():.2f}")

Normalized Attention Weights:
tensor([[0.2823, 0.1514, 0.2838, 0.1519, 0.1306],
        [0.2336, 0.1809, 0.2420, 0.1869, 0.1565],
        [0.2723, 0.1505, 0.3109, 0.1568, 0.1095],
        [0.2293, 0.1830, 0.2467, 0.1948, 0.1462],
        [0.2333, 0.1812, 0.2038, 0.1729, 0.2088]])

Sum of attention weights for the first word: 1.00


## 4. Calculating Context Vectors

Finally, we combine the original `token_embeddings` using the `attention_weights` to produce `context_vectors`. Each `context_vector` for a word is a weighted sum of all `token_embeddings`, where the weights are determined by how much attention that word pays to every other word.

This process allows each word's representation to be enriched with information from its context.

We will demonstrate this using explicit loops and then show the more efficient `torch.matmul` equivalent.

In [17]:
# Initialize a tensor to store the context vectors calculated manually.
# The shape will be (num_tokens x embedding_dim).
context_vectors_manual = torch.empty(num_tokens, embedding_dim)

# Loop through each query word (each row in attention_weights)
for i in range(attention_weights.shape[0]):
    # Initialize a temporary vector for the current context vector.
    # This vector will accumulate the weighted sum of token embeddings.
    current_context_vector = torch.zeros(embedding_dim)

    # Loop through each key word (each row in token_embeddings)
    for j in range(token_embeddings.shape[0]):
        # Multiply the attention weight (scalar) by the corresponding token embedding (vector)
        # and add it to the current context vector.
        current_context_vector += attention_weights[i, j] * token_embeddings[j]

    # Store the calculated context vector for the current query word.
    context_vectors_manual[i] = current_context_vector

print("Context vectors calculated with explicit loops:")
print(context_vectors_manual)
print("\nShape of the manual context vectors tensor:")
print(context_vectors_manual.shape)

Context vectors calculated with explicit loops:
tensor([[0.6024, 0.4031, 0.3825],
        [0.5670, 0.3585, 0.3651],
        [0.6214, 0.4129, 0.3746],
        [0.5736, 0.3587, 0.3603],
        [0.5258, 0.3387, 0.3777]])

Shape of the manual context vectors tensor:
torch.Size([5, 3])


### Efficient Calculation using Matrix Multiplication

The operation above (weighted sum) is equivalent to a matrix multiplication of the attention weights matrix and the token embeddings matrix. `torch.matmul` provides a much more efficient way to perform this operation, especially for larger tensors.

In [18]:
# Calculate context vectors using matrix multiplication (more efficient).
# The attention_weights (5x5) multiplied by token_embeddings (5x3) results in a (5x3) tensor.
context_vectors_matmul = torch.matmul(attention_weights, token_embeddings)

print("Context vectors calculated with torch.matmul:")
print(context_vectors_matmul)

# We can verify that the results from the manual loop and matmul are approximately equal.
print("\nAre the results from manual loops and matmul approximately equal?")
print(torch.allclose(context_vectors_manual, context_vectors_matmul))

Context vectors calculated with torch.matmul:
tensor([[0.6024, 0.4031, 0.3825],
        [0.5670, 0.3585, 0.3651],
        [0.6214, 0.4129, 0.3746],
        [0.5736, 0.3587, 0.3603],
        [0.5258, 0.3387, 0.3777]])

Are the results from manual loops and matmul approximately equal?
True


## Conclusion

This walkthrough demonstrated the fundamental steps of a simplified self-attention mechanism: creating token embeddings, calculating attention scores, normalizing them into attention weights, and finally using these weights to compute context-aware representations (context vectors) for each word. This mechanism is crucial for models to understand the relationships between words in a sequence.